In [ ]:
import os
import subprocess
import chemiscope
import ipi
import matplotlib.pyplot as plt
import numpy as np
from ase.io import read
import nqetools as nqe
from ase.visualize import view
# This follows:
# https://atomistic-cookbook.org/examples/pi-metad/pi-metad.html

In [ ]:
# Make a directory to store everything
directory_md = "md"
directory_metamd = "metamd"
directory_metapimd = "metapimd"
n_beads = 8
timestep = 1.0
total_steps = 5000
stride = 10
temperature = 298
thermostat = 'smart_sampling_1ps_n6_w2'
md_type = "NVT-GLE"
driver_code = 'zundel'
plumed_type = "mtd-coord"

In [ ]:
atoms = read("h5o2+.xyz", index=-1)
atoms.center(vacuum=20.0)

In [ ]:
view(atoms)

In [ ]:
# Run unbiased MD
# Make sure the directory is empty
nqe.remove_directory(directory_md)
# Run the calculation
nqe.run_md(directory_md, atoms,
           driver=driver_code,
           md_type=md_type,
           n_beads=1,
           thermostat=thermostat,
           timestep=timestep,
           total_steps=total_steps,
           stride=stride,
           temperature=temperature,
           )

In [ ]:
# Run metadynamics
# Make sure the directory is empty
nqe.remove_directory(directory_metamd)
# Run the calculation
atoms = nqe.run_plumed_md(directory_metamd, atoms,
                          driver=driver_code,
                          md_type=md_type,
                          n_beads=1,
                          thermostat=thermostat,
                          timestep=timestep,
                          total_steps=total_steps,
                          stride=stride,
                          temperature=temperature,
                          plumed_type=plumed_type,
                          )

In [ ]:
view(atoms)

In [ ]:
output_data, output_desc = ipi.read_output(os.path.join(directory_metamd, "md.out"))
colvar_data = ipi.read_trajectory(os.path.join(directory_metamd, "md.colvar_0"), format="extras")[
    "d,c1.lessthan,c2.lessthan,dc,mtd.bias"
]
traj_data = ipi.read_trajectory(os.path.join(directory_metamd, "md.pos_0.xyz"))

In [ ]:
nqe.plot_time_potential_bias(output_data)

In [ ]:
nqe.plot_time_temperature(output_data)

In [ ]:
nqe.run_plumed_hills(directory_metamd)


In [ ]:

files = nqe.search_fes_files(directory_metamd)
# rearrange data and converts to Å and eV
data = np.loadtxt(os.path.join(directory_metamd, "FES3.dat"), comments="#")[:, :3]
xyz_0 = np.array([1, 1, 1])[:, np.newaxis, np.newaxis] * data.T.reshape(3, 101, 101)
data = np.loadtxt(os.path.join(directory_metamd, "FES4.dat"), comments="#")[:, :3]
xyz_1 = np.array([1, 1, 1])[:, np.newaxis, np.newaxis] * data.T.reshape(3, 101, 101)
data = np.loadtxt(os.path.join(directory_metamd, "FES5.dat"), comments="#")[:, :3]
xyz_2 = np.array([1, 1, 1])[:, np.newaxis, np.newaxis] * data.T.reshape(3, 101, 101)


In [ ]:
nqe.plot_energy_contour_series(xyz_0, xyz_1, xyz_2)